# Ansible-2 Assignment

**Task:** Write Ansible playbooks to automate the setup and configuration of a web server
(Apache or Nginx).

This notebook creates a role-based playbook that installs and configures **Nginx**, deploys a
custom `index.html`, and ensures the service is enabled/running — plus an Apache variant for
comparison.

## 1. Directory / Role Layout

In [1]:
import os

role_dirs = [
    'project/roles/nginx/tasks',
    'project/roles/nginx/templates',
    'project/roles/nginx/handlers',
]
for d in role_dirs:
    os.makedirs(d, exist_ok=True)
print("Created Ansible role skeleton under project/roles/nginx/")


Created Ansible role skeleton under project/roles/nginx/


## 2. Inventory

In [2]:
with open('project/inventory.ini', 'w') as f:
    f.write('''[webservers]
web1 ansible_host=192.168.56.10 ansible_user=ubuntu
web2 ansible_host=192.168.56.11 ansible_user=ubuntu
''')
print("Created project/inventory.ini")


Created project/inventory.ini


## 3. Nginx Role — Tasks, Template, Handler

In [3]:
with open('project/roles/nginx/tasks/main.yml', 'w') as f:
    f.write('''---
- name: Install Nginx
  apt:
    name: nginx
    state: present
    update_cache: yes

- name: Deploy custom index.html
  template:
    src: index.html.j2
    dest: /var/www/html/index.html
    owner: www-data
    group: www-data
    mode: "0644"
  notify: Restart nginx

- name: Ensure Nginx is enabled and running
  service:
    name: nginx
    state: started
    enabled: yes

- name: Open firewall for HTTP (if ufw is active)
  ufw:
    rule: allow
    port: "80"
    proto: tcp
  ignore_errors: yes
''')

with open('project/roles/nginx/templates/index.html.j2', 'w') as f:
    f.write('''<!DOCTYPE html>
<html>
<head><title>{{ site_title }}</title></head>
<body>
  <h1>{{ site_title }}</h1>
  <p>Configured automatically by Ansible on {{ ansible_hostname }}.</p>
</body>
</html>
''')

with open('project/roles/nginx/handlers/main.yml', 'w') as f:
    f.write('''---
- name: Restart nginx
  service:
    name: nginx
    state: restarted
''')
print("Created Nginx role: tasks/main.yml, templates/index.html.j2, handlers/main.yml")


Created Nginx role: tasks/main.yml, templates/index.html.j2, handlers/main.yml


## 4. Playbook (uses the role)

In [4]:
with open('project/webserver.yml', 'w') as f:
    f.write('''---
- name: Configure Nginx web servers
  hosts: webservers
  become: true
  vars:
    site_title: "Deployed via Ansible"
  roles:
    - nginx
''')
print("Created project/webserver.yml")
print("Run: ansible-playbook -i project/inventory.ini project/webserver.yml")


Created project/webserver.yml
Run: ansible-playbook -i project/inventory.ini project/webserver.yml


## 5. Apache Variant (for comparison)
If Apache is preferred instead of Nginx, the equivalent tasks are:

In [5]:
with open('project/apache_setup.yml', 'w') as f:
    f.write('''---
- name: Configure Apache web servers
  hosts: webservers
  become: true

  tasks:
    - name: Install Apache
      apt:
        name: apache2
        state: present
        update_cache: yes

    - name: Deploy custom index.html
      copy:
        content: "<h1>Deployed via Ansible (Apache)</h1>\n"
        dest: /var/www/html/index.html
      notify: Restart apache

    - name: Ensure Apache is enabled and running
      service:
        name: apache2
        state: started
        enabled: yes

  handlers:
    - name: Restart apache
      service:
        name: apache2
        state: restarted
''')
print("Created project/apache_setup.yml")
print("Run: ansible-playbook -i project/inventory.ini project/apache_setup.yml")


Created project/apache_setup.yml
Run: ansible-playbook -i project/inventory.ini project/apache_setup.yml


## 6. Verification
```bash
ansible-playbook -i project/inventory.ini project/webserver.yml --check   # dry run
ansible-playbook -i project/inventory.ini project/webserver.yml           # apply
curl http://<web1-ip>/       # should show the templated page
```